### ONNX inference

In [5]:
import os
base_path = "/mnt/e/uho_global_models_new/asr/export"
model_path = os.path.join(base_path, "ASRMODEL_Conformer-CTC-L_spe128_en-US_6.0.int8.onnx")
vocab_path = os.path.join(base_path, "ASRVOCAB_Conformer-CTC-L_spe128_en-US_6.0.txt")
audio_path = "lamport16k.wav"

In [2]:
import librosa
import numpy as np
import torchaudio

def normalize4(x, lens):
    mean = np.mean(x, axis=2, keepdims=True)
    std = np.std(x, axis=2, keepdims=True)
    return (x - mean)/(std + 1e-5)

def compute_mel4(path):
    sample_rate = 16_000
    n_fft = 512
    win_length = 400
    hop_length = 160
    preemph = 0.97
    log_zero_guard_value = float(2**-24)

    waveforms, _ = librosa.load(path, sr=sample_rate)
    waveforms_lens = np.expand_dims(waveforms.shape[0], axis=0)

    melscale_fbanks = torchaudio.functional.melscale_fbanks(
        n_fft // 2 + 1, 0, sample_rate // 2, 80, sample_rate, "slaney", "slaney"
    ).numpy()
    print(melscale_fbanks.shape)
    
    spectrogram = librosa.stft(y=waveforms, n_fft=n_fft, hop_length=hop_length, win_length=win_length, window="hann")
    print(spectrogram.shape)
    spectrogram = np.abs(spectrogram)**2
    spectrogram = np.transpose(spectrogram)
    print(spectrogram.shape)
    mul_out = np.matmul(spectrogram, melscale_fbanks)
    print(mul_out.shape)
    log_mel_spectrogram = np.log(mul_out + log_zero_guard_value)
    log_mel_spectrogram = np.expand_dims(log_mel_spectrogram, axis=0)
    print(log_mel_spectrogram.shape)
    log_mel_spectrogram = np.transpose(log_mel_spectrogram, axes=[0,2,1])
    print(log_mel_spectrogram.shape)

    features_lens = np.int64(waveforms_lens / hop_length + 1)
    return normalize4(log_mel_spectrogram, features_lens), features_lens


/home/bko/miniconda3/envs/pnc2/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
import onnxruntime as ort
session = ort.InferenceSession(model_path)

vocab = []

with open(vocab_path) as f:
    while True:
        line = f.readline()
        if not line:
            break
        
        tok, idx = line.split()
        vocab.append(tok)


In [ ]:
import onnx
import onnxruntime as ort
import librosa
import numpy as np
import time

features, mel_len = compute_mel4(audio_path)

t0 = time.time_ns()
out = session.run(
    ["logprobs"],
    { "audio_signal": features,
      "length": mel_len }
)
t1 = time.time_ns()
print("It took ", (t1-t0)/1e9, " seconds")

import itertools

print(out[0].shape)
print(out[0].dtype) # (batch, time, numVocab)
print(np.exp(out[0]).sum(axis=-1).reshape(-1)[:10])  # validate it is log_probs
indexes = out[0].argmax(axis=-1)
print(indexes.shape)
indexes = indexes.squeeze().tolist()

unique_indexes = [k for k, _ in itertools.groupby(indexes)]

s = ""
for idx in unique_indexes:
    if idx >= len(vocab) or vocab[idx] == "<blk>":  # skip blank tokens
        continue
    s += vocab[idx] #model.decoder.vocabulary[idx]

print(s)

/home/bko/miniconda3/envs/pnc2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(257, 80)
(257, 2014)
(2014, 257)
(2014, 80)
(1, 2014, 80)
(1, 80, 2014)
It took  0.733346285  seconds
(1, 504, 129)
float32
[1.0000001  1.         0.99999994 1.0000001  0.99999994 1.0000001
 1.         0.99999994 1.         0.99999994]
(1, 504)
▁my▁relationship▁with▁computers▁began▁as▁a▁programmer▁it▁never▁quite▁occurred▁to▁me▁that▁i▁was▁doing▁anything▁scientific▁until▁after▁i▁had▁published▁enough▁papers▁that▁it▁finally▁occurred▁to▁me▁my▁education▁was▁as▁a▁mathematician▁it▁was▁just▁natural▁for▁me▁to▁think▁about▁computers
